# IntelliGrade-H — TrOCR Fine-Tuning

**Model: `microsoft/trocr-large-handwritten`** — upgraded from `small` for significantly better accuracy on messy exam handwriting.

### Before running
1. `Runtime → Change runtime type → T4 GPU` (free tier) — 16 GB VRAM, fits `large` comfortably
2. Upload your dataset to Google Drive:
   ```
   My Drive/Intelligrade/datasets/handwriting/
   ├── train/
   │   ├── images/        ← one PNG per handwritten line crop
   │   └── labels.txt     ← filename<TAB>transcription, one per line
   ├── val/
   │   ├── images/
   │   └── labels.txt
   └── test/
       ├── images/
       └── labels.txt
   ```
3. Run all cells top to bottom.

### labels.txt format (tab-separated)
```
0001.png	The mitochondria is the powerhouse of the cell
0002.png	Newton second law states F equals ma
```

### Why `trocr-large` over `trocr-small`?
| Model | CER on exam handwriting | VRAM |
|---|---|---|
| trocr-small (no fine-tune) | ~20–30% | ~2 GB |
| trocr-small fine-tuned 1000 samples | ~15–20% | ~2 GB |
| trocr-large (no fine-tune) | ~15–22% | ~8 GB |
| **trocr-large fine-tuned 500 samples** | **~10–15%** | **~8 GB** |
| **trocr-large fine-tuned 1000+ samples** | **~6–11%** | **~8 GB** |

The T4 GPU on Colab free tier has 16 GB VRAM — `large` fits with room to spare.

### Expected time on T4 GPU
- 1000 samples, 15 epochs: ~35–45 minutes
- 5000 samples, 15 epochs: ~2–3 hours


## Cell 1 — Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu}  ({mem:.1f} GB VRAM)')
    if mem < 12:
        print('⚠️  < 12 GB VRAM — reduce BATCH_SIZE to 4 in Cell 5 if you get OOM')
    elif mem >= 16:
        print('✅ 16 GB+ VRAM — trocr-large fits comfortably at BATCH_SIZE=8')
else:
    print('❌ No GPU — go to Runtime → Change runtime type → T4 GPU')
    raise SystemExit('GPU required')


## Cell 2 — Mount Google Drive and verify dataset

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/Intelligrade'
MODEL_OUT  = f'{DRIVE_BASE}/models/trocr-finetuned'
LOCAL_OUT  = '/content/trocr-finetuned'   # fast local SSD for checkpoints

os.makedirs(MODEL_OUT, exist_ok=True)
os.makedirs(LOCAL_OUT, exist_ok=True)

DRIVE_DATA = f'{DRIVE_BASE}/datasets/handwriting'

all_ok = True
for split in ['train', 'val', 'test']:
    lf = os.path.join(DRIVE_DATA, split, 'labels.txt')
    if os.path.exists(lf):
        n = sum(1 for l in open(lf) if '\t' in l)
        print(f'  ✅ {split:6s}: {n:,} samples found on Drive')
    else:
        print(f'  ❌ {split:6s}: labels.txt NOT found at {lf}')
        print(f'     Upload your dataset to Google Drive first (see Cell 0)')
        all_ok = False

if not all_ok:
    raise FileNotFoundError('Fix missing splits before continuing.')

print(f'\n✅ All splits found. Model will be saved to: {MODEL_OUT}')


## Cell 3 — Install packages

In [ ]:
# Pin versions that are stable together on Colab — do not change
!pip uninstall -y transformers peft accelerate tokenizers -q
!pip install -q transformers==4.38.2 accelerate==0.27.2 tokenizers==0.15.2 \
             evaluate==0.4.1 jiwer sentencepiece

import transformers, accelerate
print('transformers:', transformers.__version__)
print('accelerate  :', accelerate.__version__)


## Cell 4 — Dataset class, augmentation and metrics

Reads `labels.txt` (tab-separated: `filename<TAB>text`).
Augmentation is applied during training only — never on val or test.


In [ ]:
import torch
import numpy as np
import evaluate
import random
from pathlib import Path
from PIL import Image, ImageEnhance, ImageFilter
from torch.utils.data import Dataset

processor = None  # set in Cell 6 after model load


class HandwritingDataset(Dataset):
    """
    Loads handwritten line crops from a directory.
    labels.txt format (tab-separated):
        0001.png\tThe mitochondria is the powerhouse of the cell
        0002.png\tF = ma (Newton second law)
    """
    def __init__(self, data_dir, proc, max_length=128, augment=False):
        self.processor  = proc
        self.max_length = max_length
        self.augment    = augment
        self.data       = []
        data_dir   = Path(data_dir)
        images_dir = data_dir / 'images'
        with open(data_dir / 'labels.txt', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if '\t' in line:
                    fname, text = line.split('\t', 1)
                    p = images_dir / fname
                    if p.exists():
                        self.data.append((str(p), text.strip()))
        print(f'  {data_dir.name:8s}: {len(self.data):,} samples')

    def __len__(self):
        return len(self.data)

    def _augment(self, image):
        # Rotation ±3° — simulates tilted writing
        if random.random() > 0.5:
            image = image.rotate(random.uniform(-3, 3), expand=True, fillcolor=255)
        # Brightness ±20% — simulates scan quality variation
        if random.random() > 0.5:
            image = ImageEnhance.Brightness(image).enhance(random.uniform(0.8, 1.2))
        # Contrast ±20%
        if random.random() > 0.5:
            image = ImageEnhance.Contrast(image).enhance(random.uniform(0.8, 1.2))
        # Slight blur — simulates different scan resolutions
        if random.random() > 0.7:
            image = image.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1.5)))
        return image

    def __getitem__(self, idx):
        img_path, text = self.data[idx]
        image = Image.open(img_path).convert('RGB')
        if self.augment:
            image = self._augment(image)
        pixel_values = self.processor(image, return_tensors='pt').pixel_values.squeeze()
        labels = self.processor.tokenizer(
            text, padding='max_length', max_length=self.max_length,
            truncation=True, return_tensors='pt'
        ).input_ids.squeeze()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {'pixel_values': pixel_values, 'labels': labels}


def compute_metrics(eval_pred):
    preds, refs = eval_pred
    vocab_size = processor.tokenizer.vocab_size
    # Clamp to valid token range — prevents OverflowError with fp16
    preds = np.clip(preds, 0, vocab_size - 1).astype(np.int32)
    refs  = np.where(refs != -100, refs, processor.tokenizer.pad_token_id)
    refs  = np.clip(refs,  0, vocab_size - 1).astype(np.int32)
    pred_str = processor.batch_decode(preds, skip_special_tokens=True)
    ref_str  = processor.batch_decode(refs,  skip_special_tokens=True)
    cer = evaluate.load('cer').compute(predictions=pred_str, references=ref_str)
    wer = evaluate.load('wer').compute(predictions=pred_str, references=ref_str)
    return {'cer': cer, 'wer': wer}


print('✅ Dataset class and metrics ready')


## Cell 5 — Training configuration

**Do not change `MODEL_NAME`** — `trocr-large-handwritten` is the correct choice for exam handwriting.
Only reduce `BATCH_SIZE` if you get a CUDA out-of-memory error.


In [ ]:
# ─── MODEL — always large for production ────────────────────────────────────
MODEL_NAME    = 'microsoft/trocr-large-handwritten'   # upgraded from small

# ─── Hyperparameters ────────────────────────────────────────────────────────
EPOCHS        = 15          # 15 is sweet spot — more risks overfitting on small data
BATCH_SIZE    = 8           # safe for T4 (16 GB) with trocr-large; reduce to 4 if OOM
LEARNING_RATE = 5e-5        # optimal for TrOCR fine-tuning — do not change
WARMUP_STEPS  = 300
GRAD_ACCUM    = 4           # effective batch = 8 x 4 = 32
AUGMENT       = True        # always True — critical for messy handwriting generalisation

print(f'Model         : {MODEL_NAME}')
print(f'Epochs        : {EPOCHS}')
print(f'Batch size    : {BATCH_SIZE}  (effective: {BATCH_SIZE * GRAD_ACCUM})')
print(f'Learning rate : {LEARNING_RATE}')
print(f'Augmentation  : {AUGMENT}')
print(f'GPU           : {torch.cuda.get_device_name(0)}')


## Cell 6 — Copy dataset from Drive to local Colab SSD

Training directly from Google Drive is ~10× slower than local SSD.
This one-time copy takes 3–5 minutes and makes every epoch significantly faster.


In [ ]:
import shutil
from pathlib import Path

src = Path('/content/drive/MyDrive/Intelligrade/datasets/handwriting')
dst = Path('/content/datasets/handwriting')

if dst.exists():
    shutil.rmtree(dst)

print('Copying dataset to local SSD (~3-5 mins)...')
shutil.copytree(src, dst)
print('✅ Copy done')

TRAIN_DIR = '/content/datasets/handwriting/train'
VAL_DIR   = '/content/datasets/handwriting/val'
TEST_DIR  = '/content/datasets/handwriting/test'

total = 0
for split in ['train', 'val', 'test']:
    n = sum(1 for l in open(dst / split / 'labels.txt') if '\t' in l)
    print(f'  {split:6s}: {n:,} samples')
    total += n
print(f'  Total : {total:,} samples')

if total < 200:
    print('\n⚠️  Very few samples — results will be limited. Aim for 800+ train samples.')
elif total < 800:
    print(f'\nℹ️  {total} samples — expect CER ~12–18%. More data improves results.')
else:
    print(f'\n✅ {total} samples — should achieve CER < 12% after fine-tuning.')


## Cell 7 — Load model and train

Logs CER and WER after each epoch.
Early stopping halts training automatically if val CER stops improving for 4 epochs.
Best model checkpoint is restored at the end — the final saved model is always the best one.


In [ ]:
import os
from pathlib import Path
from transformers import (
    TrOCRProcessor, VisionEncoderDecoderModel,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    default_data_collator, EarlyStoppingCallback
)

LOCAL_OUT = '/content/trocr-finetuned'
MODEL_OUT = '/content/drive/MyDrive/Intelligrade/models/trocr-finetuned'
os.makedirs(LOCAL_OUT, exist_ok=True)
os.makedirs(MODEL_OUT, exist_ok=True)

# ── Verify local SSD copies ───────────────────────────────────────────────
for split, path in [('train', TRAIN_DIR), ('val', VAL_DIR), ('test', TEST_DIR)]:
    lf = Path(path) / 'labels.txt'
    if lf.exists():
        n = sum(1 for l in open(lf) if '\t' in l)
        print(f'  ✅ {split:6s}: {n:,} samples')
    else:
        raise FileNotFoundError(f'{split} not found — run Cell 6 first')

# ── Load processor and model ──────────────────────────────────────────────
print(f'\nLoading {MODEL_NAME}...')
processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model     = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.vocab_size             = model.config.decoder.vocab_size
model.gradient_checkpointing_enable()  # saves VRAM — essential for large model on T4

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'✅ Model loaded | {n_params:.0f}M parameters\n')

# ── Load datasets ─────────────────────────────────────────────────────────
print('Loading datasets from local SSD...')
train_ds = HandwritingDataset(TRAIN_DIR, processor, augment=AUGMENT)
val_ds   = HandwritingDataset(VAL_DIR,   processor, augment=False)

# ── Training arguments ────────────────────────────────────────────────────
training_args = Seq2SeqTrainingArguments(
    output_dir                  = LOCAL_OUT,
    evaluation_strategy         = 'epoch',
    save_strategy               = 'epoch',
    save_total_limit            = 2,            # keep only 2 checkpoints
    learning_rate               = LEARNING_RATE,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    weight_decay                = 0.01,
    warmup_steps                = WARMUP_STEPS,
    num_train_epochs            = EPOCHS,
    predict_with_generate       = True,
    generation_max_length       = 128,
    fp16                        = True,         # mixed precision — saves VRAM
    logging_steps               = 50,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'cer',
    greater_is_better           = False,        # lower CER = better
    remove_unused_columns       = False,
    dataloader_num_workers      = 2,
    report_to                   = [],           # disable wandb/tensorboard noise
)

trainer = Seq2SeqTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    data_collator   = default_data_collator,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=4)],
)

print('🚀 Training started...')
print('   Target: val CER < 0.12 (12%). Early stopping if no improvement for 4 epochs.\n')
trainer.train()
print('\n✅ Training complete! Best model checkpoint restored.')


## Cell 8 — Evaluate on test set

In [ ]:
import numpy as np
from torch.utils.data import DataLoader

device      = 'cuda' if torch.cuda.is_available() else 'cpu'
test_ds     = HandwritingDataset(TEST_DIR, processor, augment=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

cer_metric = evaluate.load('cer')
wer_metric = evaluate.load('wer')
all_preds, all_refs = [], []

model.to(device)
model.eval()

with torch.no_grad():
    for batch in test_loader:
        px      = batch['pixel_values'].to(device)
        gen_ids = model.generate(px, max_length=128, num_beams=4, early_stopping=True)
        preds   = processor.batch_decode(gen_ids, skip_special_tokens=True)
        labels  = batch['labels'].cpu().numpy()
        labels  = np.where(labels != -100, labels, processor.tokenizer.pad_token_id)
        labels  = np.clip(labels, 0, processor.tokenizer.vocab_size - 1).astype(np.int32)
        refs    = processor.batch_decode(labels, skip_special_tokens=True)
        all_preds.extend(preds)
        all_refs.extend(refs)

cer = cer_metric.compute(predictions=all_preds, references=all_refs)
wer = wer_metric.compute(predictions=all_preds, references=all_refs)

print('\n' + '='*52)
print('   IntelliGrade-H — Final Test Results')
print('='*52)
print(f'  Character Error Rate (CER): {cer*100:.2f}%')
print(f'  Word Error Rate (WER)     : {wer*100:.2f}%')
print(f'  Samples evaluated         : {len(test_ds):,}')
print('='*52)

if cer < 0.08:
    print('✅ Excellent! CER < 8% — competitive with Google Vision on exam data.')
elif cer < 0.12:
    print('✅ Good. CER < 12% — significantly better than base trocr-large.')
elif cer < 0.20:
    print('ℹ️  Acceptable. Consider collecting more messy handwriting samples.')
else:
    print('⚠️  CER > 20%. Collect more data — aim for 1000+ training samples.')


## Cell 9 — Save model to Google Drive

In [ ]:
import shutil
from pathlib import Path

LOCAL_OUT = '/content/trocr-finetuned'
MODEL_OUT = '/content/drive/MyDrive/Intelligrade/models/trocr-finetuned'

trainer.save_model(LOCAL_OUT)
processor.save_pretrained(LOCAL_OUT)

print(f'Copying model to Drive: {MODEL_OUT}')
if Path(MODEL_OUT).exists():
    shutil.rmtree(MODEL_OUT)
shutil.copytree(LOCAL_OUT, MODEL_OUT)

# Verify — IntelliGrade-H detects the model by checking for config.json
config_ok = (Path(MODEL_OUT) / 'config.json').exists()
print(f'\n✅ Model saved to Google Drive!')
print(f'   config.json present: {config_ok}  ← IntelliGrade-H auto-detects via this file')
print(f'   Location: {MODEL_OUT}')
print("""
── Deploy to IntelliGrade-H ────────────────────────────────────────────
1. In Google Drive: right-click trocr-finetuned/ → Download as ZIP
   Path: My Drive/Intelligrade/models/trocr-finetuned/

2. Extract to your project:
   IntelliGrade-H/models/trocr-finetuned/
   (the folder must contain config.json — that triggers auto-detection)

3. Restart the backend:
   python run.py

4. The system will log:
   Fine-tuned TrOCR model found at models/trocr-finetuned — using it.

No .env changes needed — ocr_module.py detects the model automatically.
─────────────────────────────────────────────────────────────────────────
""")


## Cell 10 — Visual test (optional)

Picks 5 random samples from the val set and shows prediction vs ground truth side by side.


In [ ]:
import random
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

samples = [l.strip().split('\t') for l in open(Path(VAL_DIR) / 'labels.txt') if '\t' in l]
random.shuffle(samples)
test_cases = samples[:5]

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.eval()

fig, axes = plt.subplots(len(test_cases), 1, figsize=(14, len(test_cases) * 2.5))
if len(test_cases) == 1:
    axes = [axes]

all_cer = []
for ax, (fname, ground_truth) in zip(axes, test_cases):
    img_path = Path(VAL_DIR) / 'images' / fname
    img = Image.open(img_path).convert('RGB')
    px  = processor(img, return_tensors='pt').pixel_values.to(device)
    with torch.no_grad():
        ids  = model.generate(px, max_length=128, num_beams=4)
    pred = processor.batch_decode(ids, skip_special_tokens=True)[0]

    sample_cer = evaluate.load('cer').compute(predictions=[pred], references=[ground_truth])
    all_cer.append(sample_cer)
    icon = '✅' if sample_cer < 0.10 else ('⚠️' if sample_cer < 0.25 else '❌')

    ax.imshow(img, cmap='gray')
    ax.set_title(
        f'{icon} GT: {ground_truth}\n   PR: {pred}  |  CER: {sample_cer*100:.1f}%',
        fontsize=9, loc='left'
    )
    ax.axis('off')

plt.tight_layout()
plt.savefig('/content/sample_predictions.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Average CER on these {len(test_cases)} samples: {sum(all_cer)/len(all_cer)*100:.1f}%')
print('Figure saved to /content/sample_predictions.png')
